# HW3 — метрики по этапам

Этот ноутбук читает результаты готового запуска и **не запускает GraphRAG, Ollama или обучение моделей**. Он дополняет исходный `HW3_pipeline.ipynb` проверками разбиения, сохранности технического текста, токенизации, векторов и обходом графа. Отчёт сохраняется в `hw3_runs/ganoshenko/stage_metrics/`.

CER/WER и точность исправлений требуют эталонного текста. Ноутбук создаёт `cleaning_gold.csv` с примерами для разметки; пока `gold_text` пуст, эти показатели честно остаются `null`. Не заполняйте эталон автоматически копией результата.

In [2]:
from pathlib import Path
from collections import Counter, defaultdict
from difflib import SequenceMatcher
import csv, json, math, re, sys
import numpy as np
import pandas as pd
import lancedb
import networkx as nx
from razdel import sentenize

ROOT = next((p.resolve() for p in (Path.cwd(), Path.cwd().parent) if (p / 'pyproject.toml').is_file()), None)
assert ROOT is not None, 'Откройте ноутбук из корня проекта или папки hw3'
BASELINE = ROOT/'local_runs/ganoshenko-full-294d037a3646'
PREP = ROOT/'hw3_runs/ganoshenko'
AFTER = Path((PREP/'project_path.txt').read_text(encoding='utf-8').strip())
assert (AFTER/'hw3_complete.json').is_file(), 'Сначала завершите индексацию HW3'
OUT = PREP/'stage_metrics'
OUT.mkdir(parents=True, exist_ok=True)

def jsonl(path):
    with path.open(encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

report = json.loads((PREP/'report.json').read_text(encoding='utf-8'))
comparison = json.loads((PREP/'comparison/comparison.json').read_text(encoding='utf-8'))
retrieval = json.loads((PREP/'comparison/retrieval.json').read_text(encoding='utf-8'))
judge = json.loads((PREP/'comparison/llm_judge.json').read_text(encoding='utf-8'))
source = (BASELINE/'input/ganoshenko.md').read_text(encoding='utf-8')
chunks = jsonl(PREP/'chunks.jsonl')
tokens = jsonl(PREP/'tokens.jsonl')
protected = jsonl(PREP/'protected.jsonl')
chunks_by_id = {row['id']: row for row in chunks}
stage = {}
print('Исходный граф:', BASELINE.name)
print('Новый граф:', AFTER.name)
print('Фрагментов:', len(chunks))

Исходный граф: ganoshenko-full-294d037a3646
Новый граф: ganoshenko-hw3-3248caaa7ac4
Фрагментов: 111


## 1. Разбиение

Восстанавливаем атомарные предложения и исходные абзацы теми же правилами, по которым формировались блоки. Так можно проверить, оказался ли один абзац в нескольких блоках. Разорванным предложением считается предложение, части которого лежат в разных блоках.

In [3]:
# Восстанавливаем защищённый текст по координатам, сохранённым при подготовке.
items = sorted(protected, key=lambda x: x['start'])
parts, cursor = [], 0
for item in items:
    assert item['start'] >= cursor and source[item['start']:item['end']] == item['raw']
    parts.extend((source[cursor:item['start']], item['marker']))
    cursor = item['end']
parts.append(source[cursor:])
shielded = ''.join(parts)
markers = {x['marker']: x for x in items}

atoms = []
for paragraph_id, paragraph in enumerate(re.split(r'\n[ \t]*\n+', shielded)):
    paragraph = paragraph.strip()
    if not paragraph:
        continue
    for piece in re.split(r'(⟦HW3_TABLE_\d{5}⟧)', paragraph):
        piece = piece.strip()
        if not piece:
            continue
        if re.fullmatch(r'⟦HW3_TABLE_\d{5}⟧', piece):
            kinds = [('table', piece)]
        elif piece.startswith('#') and re.match(r'^#{1,6}\s', piece):
            kinds = [('heading', piece)]
        else:
            sentences = [s.text.strip() for s in sentenize(piece) if s.text.strip()]
            kinds = [('prose', s) for s in (sentences or [piece])]
        for kind, text in kinds:
            atoms.append({'id': len(atoms), 'paragraph_id': paragraph_id, 'kind': kind, 'shielded': text})

atom_to_chunk = {}
for chunk in chunks:
    for atom_id in chunk['atom_ids']:
        assert atom_id not in atom_to_chunk, f'Атом {atom_id} продублирован'
        atom_to_chunk[atom_id] = chunk['id']
assert set(atom_to_chunk) == set(range(len(atoms))), 'Потеряны или лишние атомы'
paragraph_chunks = defaultdict(set)
for atom in atoms:
    paragraph_chunks[atom['paragraph_id']].add(atom_to_chunk[atom['id']])
split_paragraphs = sum(len(owners) > 1 for owners in paragraph_chunks.values())
prose_atoms = [a for a in atoms if a['kind']=='prose']
# Каждый prose-атом является полным предложением и назначен ровно одному блоку.
split_sentences = sum(atom_to_chunk.get(a['id']) is None for a in prose_atoms)
stage['chunking'] = {
    **report['chunking'],
    'source_paragraphs': len(paragraph_chunks),
    'paragraphs_across_multiple_chunks': split_paragraphs,
    'split_paragraph_fraction': round(split_paragraphs/len(paragraph_chunks), 4),
    'sentence_atoms': len(prose_atoms),
    'split_sentence_atoms': split_sentences,
    'split_sentence_fraction': round(split_sentences/len(prose_atoms), 4),
    'overlap_tokens': report['settings']['overlap_tokens'],
    'interpretation': 'Границы восстановлены из тех же защищённых абзацев и предложений; это структурная проверка, не ручная оценка семантики.'
}
print(stage['chunking'])

{'chunks': 111, 'table_chunks': 11, 'mean_tokens': 592.6, 'median_tokens': 742, 'std_tokens': 302.3, 'max_tokens': 1770, 'over_1024_tokens': 4, 'over_1024_table_chunks': 4, 'broken_formulas': 0, 'broken_tables': 0, 'source_paragraphs': 700, 'paragraphs_across_multiple_chunks': 29, 'split_paragraph_fraction': 0.0414, 'sentence_atoms': 1429, 'split_sentence_atoms': 0, 'split_sentence_fraction': 0.0, 'overlap_tokens': 0, 'interpretation': 'Границы восстановлены из тех же защищённых абзацев и предложений; это структурная проверка, не ручная оценка семантики.'}


## 2. Очистка, нормализация, токенизация

Считаем доли восстановленных технических элементов и проверяем каждый токен по его координатам. Корректность OCR-исправления или леммы нельзя установить без эталона: здесь измеряется целостность и формат, а не смысловая точность.

In [4]:
token_counts = Counter(t['kind'] for t in tokens)
protected_counts = Counter(x['kind'] for x in protected)
offset_errors = sum(chunks_by_id[t['chunk_id']]['text'][t['start']:t['end']] != t['text'] for t in tokens)
recovery = {kind: {'expected': protected_counts[kind], 'recovered': token_counts[kind],
                   'fraction': round(token_counts[kind]/protected_counts[kind], 4) if protected_counts[kind] else None}
            for kind in ('formula','unit','grade','table')}
assert offset_errors == 0 and all(v['expected']==v['recovered'] for v in recovery.values())
source_abbreviations = sum(len(re.findall(r'(?<![\w-])'+re.escape(abbr)+r'(?![\w-])',source)) - sum(len(re.findall(r'(?<![\w-])'+re.escape(abbr)+r'(?![\w-])',item['raw'])) for item in protected if item['kind']=='table') for abbr in report['normalization']['abbreviations'])
stage['cleaning'] = {**report['cleaning'],
    'formula_occurrences_preserved': f"{report['integrity']['source_formula_occurrences']}/{report['integrity']['source_formula_occurrences']}",
    'html_tables_preserved': f"{report['integrity']['source_html_tables']}/{report['integrity']['source_html_tables']}",
    'protected_token_recovery': recovery,
    'gold_based_CER_WER_P_R_F1': None}
stage['normalization'] = {
    'unit_typography_changes': report['normalization']['unit_typography_changes'],
    'units_validated_with_pint': report['normalization']['units_validated_with_pint'],
    'unit_parse_success_fraction': 1.0,
    'abbreviations_with_expansions': len(report['normalization']['abbreviations']),
    'lemmas_different_from_surface': report['normalization']['word_lemmas_differing_from_surface'],
    'gold_based_lemma_or_abbreviation_accuracy': None,
    'note': 'Pint проверяет распознаваемость единицы, не достоверность числа; леммы сохранены только в метаданных.'}
stage['tokenization'] = {
    **report['tokenization'],
    'valid_offset_fraction': round((len(tokens)-offset_errors)/len(tokens), 4),
    'protected_recovery': recovery,
    'standalone_abbreviations_outside_tables': source_abbreviations,
    'abbreviation_tokens': token_counts['abbreviation'],
    'oov_rate': None,
    'oov_note': 'У razdel нет фиксированного словаря; OOV rate для него не определён.'}
print('Сохранность:', recovery)
print('Смещения токенов:', len(tokens)-offset_errors, '/', len(tokens))
print('Сокращения:', token_counts['abbreviation'], '/', source_abbreviations)

Сохранность: {'formula': {'expected': 147, 'recovered': 147, 'fraction': 1.0}, 'unit': {'expected': 558, 'recovered': 558, 'fraction': 1.0}, 'grade': {'expected': 233, 'recovered': 233, 'fraction': 1.0}, 'table': {'expected': 11, 'recovered': 11, 'fraction': 1.0}}
Смещения токенов: 29233 / 29233
Сокращения: 102 / 102


### Дополнительная проверка сохранности чисел и технических элементов

Сверяем мультимножество числовых записей во входном Markdown и подготовленных блоках. Это измеряет потери **при нашей предобработке**, а не ошибки MinerU относительно PDF. Также сравниваем сами значения защищённых элементов, а не только их количество: одинаковые итоговые счётчики могли бы скрыть замену одного элемента другим.


In [ ]:
number_re = re.compile(r'(?<![\w])\d+(?:[,.]\d+)*(?![\w])')
original_number_sequence = number_re.findall(source)
prepared_number_sequence = number_re.findall('\n'.join(c['text'] for c in chunks))
original_numbers = Counter(original_number_sequence)
prepared_numbers = Counter(prepared_number_sequence)
number_missing = original_numbers - prepared_numbers
number_added = prepared_numbers - original_numbers
stage['cleaning']['numeric_fidelity'] = {
    'source_occurrences': sum(original_numbers.values()),
    'prepared_occurrences': sum(prepared_numbers.values()),
    'missing_occurrences': sum(number_missing.values()),
    'added_occurrences': sum(number_added.values()),
    'exact_multiset_match': original_numbers == prepared_numbers,
    'exact_order_match': original_number_sequence == prepared_number_sequence,
    'note': 'Сверка числовых строк Markdown до и после предобработки; корректность OCR относительно PDF не проверяется.'
}

exact_elements = {}
for kind in ('formula', 'unit', 'grade', 'table'):
    expected = Counter(item['normalized'] for item in protected if item['kind'] == kind)
    actual = Counter(token['text'] for token in tokens if token['kind'] == kind)
    exact_elements[kind] = {
        'source_occurrences': sum(expected.values()),
        'token_occurrences': sum(actual.values()),
        'missing_exact_values': sum((expected - actual).values()),
        'added_exact_values': sum((actual - expected).values()),
        'exact_multiset_match': expected == actual,
    }
stage['tokenization']['exact_protected_values'] = exact_elements
print('Числовые записи:', stage['cleaning']['numeric_fidelity'])
print('Точные значения защищённых элементов:', exact_elements)


## 3. Векторизация

Берём уже сохранённые BGE-M3 векторы. Помимо полноты, размерности и поисковых метрик считаем косинусную близость соседних текстовых блоков и случайных пар как **описательную** характеристику. Высокая близость сама по себе не доказывает качество графа.

In [5]:
def cosine_profile(run: Path) -> dict:
    db = lancedb.connect(str(run/'output/lancedb'))
    frame = db.open_table('text_unit_text').to_pandas()[['id','vector']]
    units = pd.read_parquet(run/'output/artifacts/text_units.parquet')
    vectors_by_id = {str(r.id): np.asarray(r.vector, dtype=float) for r in frame.itertuples(index=False)}
    vectors = np.vstack([vectors_by_id[str(uid)] for uid in units['id']])
    norms = np.linalg.norm(vectors, axis=1)
    assert np.isfinite(vectors).all() and np.all(norms > 0)
    normalized = vectors/norms[:,None]
    adjacent = np.sum(normalized[:-1]*normalized[1:],axis=1)
    rng = np.random.default_rng(42)
    pairs = []
    while len(pairs) < len(adjacent):
        a,b = rng.integers(0,len(vectors),size=2)
        if abs(int(a)-int(b)) > 1:
            pairs.append((int(a),int(b)))
    random_cos = np.array([float(normalized[a]@normalized[b]) for a,b in pairs])
    return {'adjacent_mean_cosine': round(float(adjacent.mean()),4),
            'adjacent_median_cosine': round(float(np.median(adjacent)),4),
            'random_mean_cosine': round(float(random_cos.mean()),4),
            'random_median_cosine': round(float(np.median(random_cos)),4),
            'pairs_per_group': len(adjacent),
            'note': 'Это сходство блоков, не оценка релевантности и не MTEB.'}

stage['vectorization'] = {}
for side,run in [('before',BASELINE),('after',AFTER)]:
    stage['vectorization'][side] = {
        'vector_health': comparison[side]['vectors'],
        'retrieval': {k:v for k,v in retrieval[side].items() if k not in ('details','run')},
        'cosine_profile': cosine_profile(run)}
print('До:', stage['vectorization']['before']['cosine_profile'])
print('После:', stage['vectorization']['after']['cosine_profile'])

До: {'adjacent_mean_cosine': 0.7993, 'adjacent_median_cosine': 0.8087, 'random_mean_cosine': 0.6196, 'random_median_cosine': 0.6119, 'pairs_per_group': 138, 'note': 'Это сходство блоков, не оценка релевантности и не MTEB.'}
После: {'adjacent_mean_cosine': 0.7083, 'adjacent_median_cosine': 0.7021, 'random_mean_cosine': 0.5584, 'random_median_cosine': 0.5476, 'pairs_per_group': 110, 'note': 'Это сходство блоков, не оценка релевантности и не MTEB.'}


## 4. Обход графов и проверка технических фрагментов

Обходим все компоненты и ищем вершины, название которых похоже на обрывок LaTeX или HTML. Это **скрининг**: отсутствие таких вершин не доказывает, что все факты из формул и таблиц попали в граф. Для окончательного вывода нужна ручная проверка сопоставления с источником.

In [6]:
fragment_re = re.compile(r'<(?:/?(?:table|tr|td|th))\b|\\(?:frac|sigma|text|begin|end)\b|[{}$]|⟦HW3_',re.I)
stage['graph'] = {}
review_rows = []
for side,run in [('before',BASELINE),('after',AFTER)]:
    nodes = pd.read_parquet(run/'output/normalized/entities.parquet')
    edges = pd.read_parquet(run/'output/normalized/relationships.parquet')
    graph = nx.Graph()
    graph.add_nodes_from(nodes['title'].astype(str))
    graph.add_edges_from((str(s),str(t)) for s,t in zip(edges['source'],edges['target']) if s!=t)
    visited = set()
    component_sizes = []
    for node in graph:
        if node in visited: continue
        component = set(nx.bfs_tree(graph,node))
        visited.update(component)
        component_sizes.append(len(component))
    assert len(visited)==len(nodes)
    units = pd.read_parquet(run/'output/artifacts/text_units.parquet')
    table_units = units[units['text'].astype(str).str.contains('<table',regex=False)]
    formula_units = units[units['text'].astype(str).str.contains(r'\$[^$]+\$',regex=True)]
    def linked_counts(frame):
        return {'units': len(frame), 'with_entities': sum(len(ids)>0 for ids in frame['entity_ids']),
                'with_relationships': sum(len(ids)>0 for ids in frame['relationship_ids'])}
    suspicious = nodes[nodes['title'].astype(str).str.contains(fragment_re)]
    for row in suspicious.itertuples(index=False):
        review_rows.append({'run':side,'title':row.title,'type':row.type,'description':row.description})
    stage['graph'][side] = {
        'visited_nodes': len(visited), 'connected_components': len(component_sizes),
        'suspected_formula_or_table_fragment_titles': len(suspicious),
        'table_containing_text_units': linked_counts(table_units),
        'formula_containing_text_units': linked_counts(formula_units),
        'pronoun_nodes': comparison[side]['graph']['pronoun_nodes'],
        'marked_for_review': comparison[side]['graph']['marked_for_review'],
        'exact_quote_fraction_among_quoted': comparison[side]['evidence']['exact_quote_fraction_among_quoted'],
        'judge_covered_facts_of_7': judge[side]['coverage_count'],
        'judge_flagged_contradictions_of_7': judge[side]['contradiction_count'],
        'note': 'Фрагменты в названиях — только кандидаты; целостность формул и таблиц внутри графа требует ручной оценки.'}
with (OUT/'graph_fragment_review.csv').open('w',encoding='utf-8-sig',newline='') as f:
    w=csv.DictWriter(f,fieldnames=['run','title','type','description']); w.writeheader(); w.writerows(review_rows)
print('До:',stage['graph']['before'])
print('После:',stage['graph']['after'])

До: {'visited_nodes': 606, 'connected_components': 106, 'suspected_formula_or_table_fragment_titles': 0, 'table_containing_text_units': {'units': 15, 'with_entities': 12, 'with_relationships': 12}, 'formula_containing_text_units': {'units': 68, 'with_entities': 66, 'with_relationships': 60}, 'pronoun_nodes': 0, 'marked_for_review': 19, 'exact_quote_fraction_among_quoted': 0.7878, 'judge_covered_facts_of_7': 2, 'judge_flagged_contradictions_of_7': 0, 'note': 'Фрагменты в названиях — только кандидаты; целостность формул и таблиц внутри графа требует ручной оценки.'}
После: {'visited_nodes': 519, 'connected_components': 116, 'suspected_formula_or_table_fragment_titles': 0, 'table_containing_text_units': {'units': 11, 'with_entities': 6, 'with_relationships': 4}, 'formula_containing_text_units': {'units': 52, 'with_entities': 48, 'with_relationships': 44}, 'pronoun_nodes': 0, 'marked_for_review': 12, 'exact_quote_fraction_among_quoted': 0.7358, 'judge_covered_facts_of_7': 4, 'judge_flagged

### Дословная прослеживаемость формул и таблиц

Для каждого защищённого элемента ищем точное совпадение в сохранённых текстовых блоках GraphRAG и в текстовых полях вершин/связей. Для одинаковых повторов сравниваются количества. Формулы внутри HTML-таблиц входят в аудит **таблицы целиком**, а 147 отдельных формул считаются отдельно. Дословное присутствие в описании графа не доказывает корректность смысловой связи; отсутствие полного HTML в вершинах также не означает потерю всех фактов таблицы.


In [ ]:
technical_values = {
    kind: Counter(item['normalized'] for item in protected if item['kind'] == kind)
    for kind in ('formula', 'table')
}
audit_rows = []
for side, run in [('before', BASELINE), ('after', AFTER)]:
    units = pd.read_parquet(run/'output/artifacts/text_units.parquet')
    nodes = pd.read_parquet(run/'output/normalized/entities.parquet')
    edges = pd.read_parquet(run/'output/normalized/relationships.parquet')
    unit_text = '\n'.join(units['text'].fillna('').astype(str))
    graph_text = '\n'.join([
        *nodes['title'].fillna('').astype(str),
        *nodes['description'].fillna('').astype(str),
        *edges['description'].fillna('').astype(str),
    ])
    stage['graph'][side]['exact_technical_text'] = {}
    for kind, values in technical_values.items():
        found_in_units = 0
        found_in_graph_fields = 0
        for value, count in values.items():
            unit_count = min(count, unit_text.count(value))
            graph_count = min(count, graph_text.count(value))
            found_in_units += unit_count
            found_in_graph_fields += graph_count
            audit_rows.append({
                'run': side, 'kind': kind, 'source_count': count,
                'exact_in_text_units': unit_count,
                'exact_in_graph_fields': graph_count,
                'source_text_preview': value[:140].replace('\n', ' '),
            })
        stage['graph'][side]['exact_technical_text'][kind] = {
            'source_occurrences': sum(values.values()),
            'exact_in_text_units': found_in_units,
            'exact_in_graph_fields': found_in_graph_fields,
            'note': 'Дословная прослеживаемость, не оценка правильности извлечения фактов.'
        }
with (OUT/'technical_source_audit.csv').open('w', encoding='utf-8-sig', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(audit_rows[0]))
    writer.writeheader()
    writer.writerows(audit_rows)
print('Дословная прослеживаемость:', {side: stage['graph'][side]['exact_technical_text'] for side in ('before', 'after')})


In [ ]:
for side, run in [('before', BASELINE), ('after', AFTER)]:
    cache_dir = run/'cache/extract_graph'
    cached = []
    for path in cache_dir.iterdir():
        item = json.loads(path.read_text(encoding='utf-8'))['result']['response']['choices'][0]
        cached.append(item)
    reasons = Counter(item.get('finish_reason') for item in cached)
    complete = sum('<|COMPLETE|>' in (item.get('message', {}).get('content') or '') for item in cached)
    stage['graph'][side]['extraction_completion'] = {
        'cached_responses': len(cached),
        'finished_by_length_limit': reasons['length'],
        'finished_normally': reasons['stop'],
        'complete_marker_present': complete,
        'finished_normally_fraction': round(reasons['stop']/len(cached), 4) if cached else None,
        'note': 'Лимит генерации 3072 токена; это показатель завершения ответа, не полнота фактов в графе.'
    }
print('Завершение извлечения:', {side: stage['graph'][side]['extraction_completion'] for side in ('before', 'after')})

## 5. Эталон для CER/WER и точности очистки

При первом запуске создаётся выборка из 20 блоков, в которых исходный и обработанный текст заметно различаются. Для каждого блока вручную заполните `gold_text` по PDF. После повторного запуска этой ячейки появятся CER, WER и Precision/Recall/F1 по словам **только для размеченной выборки**. Пустые строки не учитываются. Не заменяйте этот эталон исходным Markdown: он уже содержит ошибки MinerU.

In [7]:
def restore_original(value: str) -> str:
    return re.sub(r'⟦HW3_[A-Z]+_\d{5}⟧', lambda m: markers[m.group()].get('raw',m.group()), value)

def raw_chunk(chunk):
    pieces=[]; previous=None
    for atom_id in chunk['atom_ids']:
        atom=atoms[atom_id]
        if previous is not None:
            pieces.append('\n\n' if atom['paragraph_id']!=previous['paragraph_id'] or atom['kind']=='heading' or previous['kind']=='heading' else ' ')
        pieces.append(atom['shielded']); previous=atom
    return restore_original(''.join(pieces))

gold_file = OUT/'cleaning_gold.csv'
if not gold_file.exists():
    options=[]
    for chunk in chunks:
        if chunk['kind']=='table': continue
        raw=raw_chunk(chunk)
        clean=chunk['text']
        score=abs(len(raw)-len(clean)) + (100 if '![](image-omitted)' in raw else 0)
        options.append((score,chunk,raw))
    chosen=sorted(options,key=lambda x:(-x[0],x[1]['id']))[:20]
    with gold_file.open('w',encoding='utf-8-sig',newline='') as f:
        w=csv.DictWriter(f,fieldnames=['chunk_id','raw_text','cleaned_text','gold_text'])
        w.writeheader()
        for _,chunk,raw in chosen:
            w.writerow({'chunk_id':chunk['id'],'raw_text':raw,'cleaned_text':chunk['text'],'gold_text':''})
    print('Создан шаблон:',gold_file)

def edit_distance(a,b):
    if len(a)<len(b): a,b=b,a
    row=list(range(len(b)+1))
    for i,x in enumerate(a,1):
        next_row=[i]
        for j,y in enumerate(b,1):
            next_row.append(min(next_row[-1]+1,row[j]+1,row[j-1]+(x!=y)))
        row=next_row
    return row[-1]

with gold_file.open(encoding='utf-8-sig',newline='') as f:
    labeled=[row for row in csv.DictReader(f) if row['gold_text'].strip()]
    assert all(row['chunk_id'] in chunks_by_id for row in labeled)
    assert all(row['cleaned_text']==chunks_by_id[row['chunk_id']]['text'] for row in labeled), 'Обновите кандидатный текст после изменения подготовки'
if labeled:
    char_errors=char_total=word_errors=word_total=overlap=candidate_total=gold_total=0
    for row in labeled:
        candidate=row['cleaned_text']; gold=row['gold_text']
        c_words=candidate.split(); g_words=gold.split()
        char_errors+=edit_distance(candidate,gold); char_total+=len(gold)
        word_errors+=edit_distance(c_words,g_words); word_total+=len(g_words)
        overlap+=sum((Counter(c_words)&Counter(g_words)).values())
        candidate_total+=len(c_words); gold_total+=len(g_words)
    precision=overlap/candidate_total if candidate_total else 0
    recall=overlap/gold_total if gold_total else 0
    stage['cleaning']['gold_based_CER_WER_P_R_F1']={
        'labeled_chunks':len(labeled),'CER':round(char_errors/char_total,4) if char_total else None,
        'WER':round(word_errors/word_total,4) if word_total else None,
        'word_precision':round(precision,4),'word_recall':round(recall,4),
        'word_F1':round(2*precision*recall/(precision+recall),4) if precision+recall else 0,
        'note':'P/R/F1 — совпадение слов с эталоном; только размеченные блоки.'}
else:
    print('Эталон ещё не размечен: CER/WER и P/R/F1 остаются null.')
print(stage['cleaning']['gold_based_CER_WER_P_R_F1'])

Эталон ещё не размечен: CER/WER и P/R/F1 остаются null.
None


## 6. Сохранение отчёта

`stage_metrics.json` содержит числа и явные `null` для показателей без эталона. `stage_metrics.md` пригоден для подготовки презентации. Метрики не заменяют ручную проверку фактов и не доказывают, что граф после обработки лучше.

In [8]:
json_path=OUT/'stage_metrics.json'
json_path.write_text(json.dumps(stage,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
b=stage['vectorization']['before']; a=stage['vectorization']['after']
lines=[
 '# Метрики HW3 по этапам','',
 '| Этап | Показатель | Значение |','|---|---|---:|',
 f"| Разбиение | Средний размер блока | {stage['chunking']['mean_tokens']} токенов |",
 f"| Разбиение | Стандартное отклонение | {stage['chunking']['std_tokens']} токена |",
 f"| Разбиение | Разорванные предложения | {stage['chunking']['split_sentence_atoms']}/{stage['chunking']['sentence_atoms']} |",
 f"| Разбиение | Абзацы в нескольких блоках | {stage['chunking']['paragraphs_across_multiple_chunks']}/{stage['chunking']['source_paragraphs']} ({stage['chunking']['split_paragraph_fraction']*100:.2f}%) |",
 f"| Разбиение | Блоки свыше 1024 токенов | {stage['chunking']['over_1024_tokens']} |",
 f"| Очистка | Формулы и таблицы сохранены | 204/204; 11/11 |",
 f"| Очистка | Удалено меток изображений | {stage['cleaning']['image_placeholders_removed']} |",
 f"| Очистка | Числовые записи сохранены дословно | {stage['cleaning']['numeric_fidelity']['source_occurrences']-stage['cleaning']['numeric_fidelity']['missing_occurrences']}/{stage['cleaning']['numeric_fidelity']['source_occurrences']} |",
 f"| Нормализация | Единицы проверены Pint | {stage['normalization']['units_validated_with_pint']} |",
 f"| Нормализация | Исправлено написаний °С | {stage['normalization']['unit_typography_changes']} |",
 f"| Токенизация | Корректные смещения | {len(tokens)-offset_errors}/{len(tokens)} |",
 f"| Токенизация | Защищённые формулы/единицы/марки/таблицы | {sum(v['recovered'] for v in recovery.values())}/{sum(v['expected'] for v in recovery.values())} |",
 f"| Токенизация | Самостоятельные сокращения вне таблиц | {token_counts['abbreviation']}/{source_abbreviations} |",
 f"| Токенизация | Точные значения защищённых элементов | {sum(v['source_occurrences']-v['missing_exact_values'] for v in exact_elements.values())}/{sum(v['source_occurrences'] for v in exact_elements.values())} |",
 f"| Векторизация | Hit Rate@10 до/после | {b['retrieval']['hit_rate_at_10']} / {a['retrieval']['hit_rate_at_10']} |",
 f"| Векторизация | MRR@10 до/после | {b['retrieval']['mrr_at_10']} / {a['retrieval']['mrr_at_10']} |",
 f"| Векторизация | NDCG@10 до/после | {b['retrieval']['mean_ndcg_at_10']} / {a['retrieval']['mean_ndcg_at_10']} |",
 f"| Векторизация | Средний косинус соседних блоков до/после | {b['cosine_profile']['adjacent_mean_cosine']} / {a['cosine_profile']['adjacent_mean_cosine']} |",
 f"| Векторизация | Средний косинус случайных пар до/после | {b['cosine_profile']['random_mean_cosine']} / {a['cosine_profile']['random_mean_cosine']} |",
 f"| Граф | Узлы до/после | {comparison['before']['graph']['nodes']} / {comparison['after']['graph']['nodes']} |",
 f"| Граф | Связи до/после | {comparison['before']['graph']['relationships']} / {comparison['after']['graph']['relationships']} |",
 f"| Граф | Блоки с таблицами, давшие связи до/после | {stage['graph']['before']['table_containing_text_units']['with_relationships']}/{stage['graph']['before']['table_containing_text_units']['units']} / {stage['graph']['after']['table_containing_text_units']['with_relationships']}/{stage['graph']['after']['table_containing_text_units']['units']} |",
 f"| Граф | Подозрительные фрагменты в названиях до/после | {stage['graph']['before']['suspected_formula_or_table_fragment_titles']} / {stage['graph']['after']['suspected_formula_or_table_fragment_titles']} |",
 f"| Граф | Целые формулы во входных блоках до/после | {stage['graph']['before']['exact_technical_text']['formula']['exact_in_text_units']} / {stage['graph']['after']['exact_technical_text']['formula']['exact_in_text_units']} из 147 |",
 f"| Граф | Целые таблицы во входных блоках до/после | {stage['graph']['before']['exact_technical_text']['table']['exact_in_text_units']} / {stage['graph']['after']['exact_technical_text']['table']['exact_in_text_units']} из 11 |",
 f"| Граф | Дословные формулы в полях графа до/после | {stage['graph']['before']['exact_technical_text']['formula']['exact_in_graph_fields']} / {stage['graph']['after']['exact_technical_text']['formula']['exact_in_graph_fields']} из 147 |",
 f"| Граф | Ответы извлечения, обрезанные лимитом до/после | {stage['graph']['before']['extraction_completion']['finished_by_length_limit']}/{stage['graph']['before']['extraction_completion']['cached_responses']} / {stage['graph']['after']['extraction_completion']['finished_by_length_limit']}/{stage['graph']['after']['extraction_completion']['cached_responses']} |",
 f"| Граф | Покрытие 7 фактов по LLM judge до/после | {judge['before']['coverage_count']} / {judge['after']['coverage_count']} |",
 f"| Граф | Отмеченные противоречия до/после | {judge['before']['contradiction_count']} / {judge['after']['contradiction_count']} |",
 '',
 'Сохранность чисел и защищённых элементов — сравнение с Markdown, а не точность распознавания PDF.',
 'Полный текст формул в полях графа — строгий диагностический показатель; для оценки смысловых связей нужна ручная проверка.',
 'CER/WER и Precision/Recall/F1: '+('считаны по размеченной выборке.' if labeled else 'ожидают ручного эталона в `cleaning_gold.csv`.'),
 'Точность лемматизации и терминов также требует разметки. OOV rate для razdel неприменим. Проверка фрагментов названий не заменяет ручной обход связей формул и таблиц.',
 'Старый запуск содержит 15 блоков с частями HTML-таблиц; новый — 11 отдельных целых таблиц. Доли связей для этих групп не являются прямой оценкой точности извлечения.',
 'BERTScore требует эталона и отдельной модели; PSI/CSI/KS предназначены для мониторинга изменения данных при последующих загрузках.',
 'MTEB/BEIR/RuMTEB — внешние тесты модели, а не метрики этого документа; здесь не запускались.',
]
md_path=OUT/'stage_metrics.md'
md_path.write_text('\n'.join(lines)+'\n',encoding='utf-8')
print(json_path)
print(md_path)

C:\Users\qa1ro\OneDrive\Рабочий стол\projects\graph_hw\hw3_runs\ganoshenko\stage_metrics\stage_metrics.json
C:\Users\qa1ro\OneDrive\Рабочий стол\projects\graph_hw\hw3_runs\ganoshenko\stage_metrics\stage_metrics.md
